# AIC y BIC a partir de las corridas de MCMC convergidas

Comparación de $\Lambda$CDM con Hu-Sawicki (HS) y Exponencial (EXP) para los dos análisis conjuntos
de `fR-MCMC_test_runs` (DE moves, 32 walkers, integrador en C, $r_d$ de CLASS en cada paso):

| Conjunto | Datos | $N$ |
|---|---|---|
| **viejo** | CC (diagonal) + Pantheon + BAO full + AGN | 3514 |
| **nuevo** | CC con covarianza de Moresco + Pantheon+ (sin SH0ES) + QSO (Benetti+ 2025) + DESI DR2 | 3647 |

- $\chi^2_{\min}$ se obtiene del mejor punto de la cadena y se refina con Nelder-Mead
  (desde ese punto y desde el máximo de verosimilitud guardado en `summary.json`), con la misma
  likelihood que las corridas (`likelihood_ext.log_likelihood`, `use_c=True`).
- $\mathrm{AIC} = \chi^2_{\min} + 2k$, $\quad \mathrm{BIC} = \chi^2_{\min} + k\ln N$, $\quad \Delta = $ modelo $-$ $\Lambda$CDM.
- $k$: $\Lambda$CDM tiene $[M_{\rm abs}, \Omega_m, H_0]$ ($k=3$); HS y EXP suman $b$ ($k=4$).
  En el conjunto nuevo se suma el offset de calibración $k_{\rm QSO}$ de QSO, marginalizado analíticamente
  (con prior plano eso equivale a minimizarlo), así que suma 1 a $k$ en los tres modelos y no cambia los $\Delta$.
  $\omega_b$ está fijo en el valor de BBN y no cuenta.
- **Radiación:** estas corridas se hicieron sin radiación en el fondo, así que se usa
  `constants.RADIATION = False` (el default del repo ahora es `True`).

## 1. Configuración

In [1]:
import os
import sys
import json

import numpy as np
import pandas as pd
import emcee
import git
from scipy.optimize import minimize

os.environ.setdefault('OMP_NUM_THREADS', '1')
path_git = git.Repo('.', search_parent_directories=True).working_tree_dir
path_runs = os.path.join(os.path.dirname(path_git), 'fR-MCMC_test_runs')
os.chdir(path_git)
sys.path.extend([os.path.join(path_git, 'fr_mcmc'), os.path.join(path_git, 'fr_mcmc', 'utils')])

import constants
constants.RADIATION = False  # como en las corridas
from likelihood_ext import log_likelihood
from data import (read_data_AGN, read_data_BAO_full, read_data_chronometers, read_data_DESI,
                  read_data_pantheon, read_data_pantheon_plus)
from QSO import read_data_QSO
from CC_cov import read_data_CC_cov

DIR_OUT = os.path.join(path_git, 'notebooks', 'output')
os.makedirs(DIR_OUT, exist_ok=True)
print('Corridas en:', path_runs)

Corridas en: /home/matias/Documents/PhD/code/fR-MCMC_test_runs


## 2. Datos

In [2]:
src = os.path.join(path_git, 'fr_mcmc', 'source')
def read(folder, func, *files):
    os.chdir(os.path.join(src, folder))
    try:
        return func(*files)
    finally:
        os.chdir(path_git)

ds_CC = read('CC', read_data_chronometers, 'chronometers_data.txt')
ds_SN = read('Pantheon', read_data_pantheon, 'lcparam_full_long_zhel.txt')
ds_AGN = read('AGN', read_data_AGN, 'table3.dat')
ds_BAO = read('BAO_full', read_data_BAO_full, 'BAO_full_1.csv', 'BAO_full_2.csv')
ds_CCcov = read('CC_cov', read_data_CC_cov, '../CC/chronometers_data.txt', 'HzTable_MM_BC03.dat', 'data_MM20.dat')
ds_PP = read('Pantheon_plus_shoes', read_data_pantheon_plus, 'Pantheon+SH0ES.dat', 'Pantheon+SH0ES_STAT+SYS.cov')
ds_QSO = read('QSO', read_data_QSO, 'qso_benetti2025.txt')
ds_DESI = read('DESI', read_data_DESI, 'DESI_DR2_dm_dh.txt', 'DESI_DR2_dv.txt')

# BAO full: medidas sueltas + pares (D_M, D_H); DESI: pares (D_M, D_H) + D_V
CONJUNTOS = {
    'viejo': dict(runs='CC+Pantheon+AGN+BAO_full',
                  kwargs=dict(dataset_CC=ds_CC, dataset_SN=ds_SN, dataset_AGN=ds_AGN, dataset_BAO_full=ds_BAO),
                  N={'CC': len(ds_CC[0]), 'Pantheon': len(ds_SN[0]), 'AGN': len(ds_AGN[0]),
                     'BAO_full': len(ds_BAO[0][0]) + 2 * len(ds_BAO[1][0])},
                  k_extra=0),
    'nuevo': dict(runs='CCcov+PPlus+QSO+DESI',
                  kwargs=dict(dataset_CC_cov=ds_CCcov, dataset_SN_plus=ds_PP, dataset_QSO=ds_QSO, dataset_DESI=ds_DESI),
                  N={'CCcov': len(ds_CCcov[0]), 'Pantheon+': len(ds_PP[0]), 'QSO': len(ds_QSO[0]),
                     'DESI DR2': 2 * len(ds_DESI[0][0]) + len(ds_DESI[1][0])},
                  k_extra=1),  # k_QSO
}
for name, c in CONJUNTOS.items():
    c['N_tot'] = sum(c['N'].values())
    print('{:6s} {}  N total = {}  ln N = {:.4f}'.format(name, c['N'], c['N_tot'], np.log(c['N_tot'])))

viejo  {'CC': 30, 'Pantheon': 1048, 'AGN': 2421, 'BAO_full': 15}  N total = 3514  ln N = 8.1645
nuevo  {'CCcov': 30, 'Pantheon+': 1590, 'QSO': 2014, 'DESI DR2': 13}  N total = 3647  ln N = 8.2017


## 3. $\chi^2_{\min}$ de cada modelo

In [3]:
MODELS = {'LCDM': dict(index=31, names=['M', 'Om', 'H0']),
          'HS':   dict(index=42, names=['M', 'Om', 'b', 'H0']),
          'EXP':  dict(index=42, names=['M', 'Om', 'b', 'H0'])}
# Priors de las corridas (run_test.py: Om < 0.4; run_test_ext.py: Om < 0.6)
BOUNDS = {'viejo': {'M': [-20, -18], 'Om': [0.05, 0.4], 'b': [0, 5], 'H0': [50, 100]},
          'nuevo': {'M': [-20, -18], 'Om': [0.05, 0.6], 'b': [0, 5], 'H0': [50, 100]}}

def chi2_min(conjunto, model):
    c, cfg = CONJUNTOS[conjunto], MODELS[model]
    run = os.path.join(path_runs, '{}_{}_DE'.format(model, c['runs']))
    summary = json.load(open(os.path.join(run, 'summary.json')))
    assert summary['converged'] and not summary.get('radiation', False), run
    reader = emcee.backends.HDFBackend(os.path.join(run, 'chain.h5'), read_only=True)
    log_prob, chain = reader.get_log_prob(flat=True), reader.get_chain(flat=True)
    x_chain = chain[np.argmax(log_prob)]
    bounds = [BOUNDS[conjunto][k] for k in cfg['names']]

    chi2 = lambda theta: -2 * log_likelihood(theta, 147, index=cfg['index'], model=model,
                                             use_c=True, use_ml=False, **c['kwargs'])
    candidates = [(-2 * log_prob.max(), x_chain)]
    for x0 in [x_chain, np.array([summary['maxlike'][k] for k in cfg['names']])]:
        opt = minimize(chi2, x0, method='Nelder-Mead', bounds=bounds,
                       options={'xatol': 1e-5, 'fatol': 1e-4, 'maxiter': 5000})
        candidates.append((opt.fun, opt.x))
    best, x = min(candidates, key=lambda t: t[0])
    return dict(k=len(cfg['names']) + c['k_extra'], chi2_min=best, chi2_chain=-2 * log_prob.max(),
                **dict(zip(cfg['names'], x)))

results = {}
for conjunto in CONJUNTOS:
    for model in MODELS:
        r = chi2_min(conjunto, model)
        results[(conjunto, model)] = r
        print('{:6s} {:4s} chi2_min = {:.3f}  (cadena: {:.3f})  en {}'.format(
            conjunto, model, r['chi2_min'], r['chi2_chain'],
            {k: round(v, 4) for k, v in r.items() if k in ['M', 'Om', 'b', 'H0']}))

viejo  LCDM chi2_min = 2220.251  (cadena: 2220.255)  en {'M': np.float64(-19.4023), 'Om': np.float64(0.3089), 'H0': np.float64(68.173)}


viejo  HS   chi2_min = 2220.251  (cadena: 2220.296)  en {'M': np.float64(-19.4023), 'Om': np.float64(0.3089), 'b': np.float64(0.0), 'H0': np.float64(68.1728)}


viejo  EXP  chi2_min = 2219.989  (cadena: 2219.996)  en {'M': np.float64(-19.4029), 'Om': np.float64(0.3143), 'b': np.float64(0.8299), 'H0': np.float64(67.4917)}


nuevo  LCDM chi2_min = 3532.429  (cadena: 3532.433)  en {'M': np.float64(-19.397), 'Om': np.float64(0.3127), 'H0': np.float64(68.737)}


nuevo  HS   chi2_min = 3530.101  (cadena: 3530.106)  en {'M': np.float64(-19.4362), 'Om': np.float64(0.3004), 'b': np.float64(0.2353), 'H0': np.float64(68.3999)}


nuevo  EXP  chi2_min = 3529.591  (cadena: 3529.604)  en {'M': np.float64(-19.4059), 'Om': np.float64(0.3427), 'b': np.float64(1.4266), 'H0': np.float64(65.6419)}


## 4. AIC y BIC

In [4]:
df = pd.DataFrame(results).T[['k', 'chi2_min', 'chi2_chain', 'M', 'Om', 'b', 'H0']]
df.index.names = ['conjunto', 'modelo']
df['k'] = df['k'].astype(int)
df['N'] = [CONJUNTOS[c]['N_tot'] for c, _ in df.index]
df['AIC'] = df['chi2_min'] + 2 * df['k']
df['BIC'] = df['chi2_min'] + df['k'] * np.log(df['N'])
for col in ['chi2_min', 'AIC', 'BIC']:
    ref = df.xs('LCDM', level='modelo')[col]
    df['Delta_' + col.replace('_min', '')] = df[col] - ref.reindex(df.index.get_level_values('conjunto')).values

df.to_csv(os.path.join(DIR_OUT, 'aic_bic_test_runs.csv'))
df.round(3)

k  chi2_min  chi2_chain       M     Om      b      H0     N  \
conjunto modelo                                                                
viejo    LCDM    3  2220.251    2220.255 -19.402  0.309    NaN  68.173  3514   
         HS      4  2220.251    2220.296 -19.402  0.309  0.000  68.173  3514   
         EXP     4  2219.989    2219.996 -19.403  0.314  0.830  67.492  3514   
nuevo    LCDM    4  3532.429    3532.433 -19.397  0.313    NaN  68.737  3647   
         HS      5  3530.101    3530.106 -19.436  0.300  0.235  68.400  3647   
         EXP     5  3529.591    3529.604 -19.406  0.343  1.427  65.642  3647   

                      AIC       BIC  Delta_chi2  Delta_AIC  Delta_BIC  
conjunto modelo                                                        
viejo    LCDM    2226.251  2244.745       0.000      0.000      0.000  
         HS      2228.251  2252.909       0.000      2.000      8.165  
         EXP     2227.989  2252.647      -0.263      1.737      7.902  
nuevo    LCDM    3540.429  3565.235       0.000      0.000      0.000  
         HS      3540.101  3571.109      -2.328     -0.328      5.874  
         EXP     3539.591  3570.600      -2.838     -0.838      5.364

## 5. Lectura

Escala de Jeffreys usual para $|\Delta\mathrm{BIC}|$: $<2$ no significativo, $2$–$6$ positivo,
$6$–$10$ fuerte, $>10$ muy fuerte. Signo positivo: favorece a $\Lambda$CDM; negativo: favorece al f(R).
Para $\Delta\mathrm{AIC}$, $|\Delta| \lesssim 2$ no distingue los modelos.

- **Conjunto viejo:** HS tiene el mínimo en $b=0$ ($\Lambda$CDM, mismo $\chi^2$), así que $\Delta\mathrm{AIC}=2$ y
  $\Delta\mathrm{BIC}=\ln N$: solo la penalización. EXP mejora el $\chi^2$ en $\sim0.3$.
- **Conjunto nuevo:** con Pantheon+ y DESI las posteriores de $b$ se separan de 0
  (`plots_derived/compare_CCcov+PPlus+QSO+DESI_models.png`) y los f(R) mejoran el ajuste:
  $\Delta\chi^2 = -2.3$ (HS, $b \approx 0.24$) y $-2.8$ (EXP, $b \approx 1.4$). No alcanza para compensar la
  penalización: $\Delta\mathrm{AIC} = -0.3$ y $-0.8$ (sin preferencia) y $\Delta\mathrm{BIC} = +5.9$ y $+5.4$
  (evidencia positiva a favor de $\Lambda$CDM, más débil que con el conjunto viejo, $\approx 8$).